# 01 · LRS3 data (CPU)
Downloads LRS3 from the Hugging Face mirror `TheNHz/ellipsis-lrs3-raw` (only the parts VALLR uses: **trainval** 10 GB and **test** 0.7 GB) and makes 96×96 mouth ROIs.

Run this notebook **4 times** (one Kaggle notebook copy each), changing the first cell:
| copy | `PART` | `SHARD` |
|---|---|---|
| lrs3-test | `'test'` | – |
| lrs3-trainval-0 | `'trainval'` | `0` |
| lrs3-trainval-1 | `'trainval'` | `1` |
| lrs3-trainval-2 | `'trainval'` | `2` |

Settings: Accelerator **None**, Internet **On**, secret **HF_TOKEN** attached. Use **Save Version → Save & Run All (Commit)**.

In [ ]:
PART = 'test'            # 'test' or 'trainval'
SHARD, NUM_SHARDS = 0, 3   # only used for trainval

In [ ]:
# --- setup: clone the project code ---
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/durgaanuhya02/MoVi-Swin-Motion--and-Viseme-Aware-Space-Time-Swin-Transformer-for-Robust-and-Efficient-Lip-Reading.git", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
# --- load a Kaggle secret into an environment variable (value is never printed) ---
def load_secret(name):
    from kaggle_secrets import UserSecretsClient
    os.environ[name] = UserSecretsClient().get_secret(name)
    print(f"{name} loaded")
load_secret('HF_TOKEN')

In [ ]:
!pip install -q mediapipe cmudict huggingface_hub
!df -h /tmp /kaggle/working; nproc

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
REPO = "TheNHz/ellipsis-lrs3-raw"
if PART == 'test':
    snapshot_download(REPO, repo_type="dataset", local_dir="/tmp/lrs3",
                      allow_patterns=["test-mattymchen/data/*.parquet"], token=os.environ["HF_TOKEN"])
    !python -m src.preprocess.parquet_to_rois --parquet /tmp/lrs3/test-mattymchen/data/*.parquet --out /kaggle/working/lrs3_test_rois
else:
    tar = hf_hub_download(REPO, "ainncy/trainval.tar.gz", repo_type="dataset",
                          local_dir="/tmp/lrs3", token=os.environ["HF_TOKEN"])
    !mkdir -p /tmp/lrs3_tv && tar -xzf {tar} -C /tmp/lrs3_tv && rm -f {tar}
    !find /tmp/lrs3_tv -maxdepth 3 -type d | head -5

In [ ]:
if PART == 'trainval':
    import glob
    # the folder whose sub-folders contain the .mp4/.txt clips
    root = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob('/tmp/lrs3_tv/**/*.mp4', recursive=True)})[0]
    print('LRS3 trainval root:', root)
    !python -m src.preprocess.build_lrs_manifest --root {root} --name lrs3/trainval --split train --val-fraction 0.02 --out /kaggle/working/lrs3_trainval_manifest.jsonl
    OUT = f'/kaggle/working/lrs3_trainval_shard{SHARD}'
    !python -m src.preprocess.extract_rois --manifest /kaggle/working/lrs3_trainval_manifest.jsonl --out {OUT} --workers 4 --detect-every 2 --shard {SHARD}/{NUM_SHARDS} 2>&1 | grep -E 'clips|eta' | tail -20
    !wc -l {OUT}/index.jsonl {OUT}/errors.jsonl; du -sh {OUT}

In [ ]:
!rm -rf /kaggle/working/repo   # keep only the data in the saved output